In [8]:
# CELL 1: Install
!pip install -q langchain==0.1.20 langchain-groq langchain-community

In [9]:
# CELL 2: Connect
import os
from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from datetime import datetime

os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
print("Connected!")

Connected!


In [10]:
# Test the connection with a real question
response = llm.invoke("What is fleet orchestration in one sentence?")
print(response.content)

Fleet orchestration refers to the process of managing and coordinating a large number of devices, vehicles, or machines, such as trucks, drones, or robots, to optimize their performance, efficiency, and productivity, often using software and data analytics to streamline operations.


In [11]:
# CELL 3: Fleet data and tools
fleet_vehicles = [
    {"id": "ZX001", "state": "Available", "battery": 85, "minutes_in_state": 5},
    {"id": "ZX002", "state": "Charging", "battery": 23, "minutes_in_state": 45},
    {"id": "ZX003", "state": "Passenger", "battery": 67, "minutes_in_state": 12},
    {"id": "ZX004", "state": "Offline", "battery": 8, "minutes_in_state": 120},
    {"id": "ZX005", "state": "Available", "battery": 91, "minutes_in_state": 3},
    {"id": "ZX006", "state": "Maintenance", "battery": 45, "minutes_in_state": 200},
    {"id": "ZX007", "state": "Passenger", "battery": 15, "minutes_in_state": 35},
    {"id": "ZX008", "state": "Charging", "battery": 78, "minutes_in_state": 8},
    {"id": "ZX009", "state": "Available", "battery": 55, "minutes_in_state": 2},
    {"id": "ZX010", "state": "Offline", "battery": 3, "minutes_in_state": 300},
]

@tool
def generate_incident_report(anomalies: str) -> str:
    """Generates a structured incident report with recommended actions for fleet operators."""
    if "No anomalies" in anomalies:
        return "FLEET STATUS: ALL SYSTEMS NORMAL"
    lines = anomalies.strip().split("\n")
    report = f"""
ZOOX FLEET INCIDENT REPORT
===========================
Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M")}
Total Vehicles Monitored: {len(fleet_vehicles)}
Anomalies Found: {len(lines)}

ANOMALIES DETECTED:
{anomalies}

RECOMMENDED ACTIONS:
- CRITICAL: Dispatch recovery team immediately
- WARNING: Monitor and prepare for mid-ride charging stop
- ALERT: Contact operations center for status update
- INFO: Free up charger for higher priority vehicles

Fleet Health Score: {round((len(fleet_vehicles) - len(lines)) / len(fleet_vehicles) * 100)}%
===========================
"""
    return report

print("Tools ready!")


Tools ready!


In [13]:
# CELL 4: Build agent
tools = [detect_anomalies, generate_incident_report]

template = '''Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}'''

prompt = PromptTemplate.from_template(template)
agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)
print("Agent ready!")

Agent ready!


In [14]:

# CELL 5: Run agent
result = agent_executor.invoke({
    "input": "Which vehicles should be prioritized for immediate action and why? What would you recommend the operations manager do in the next 10 minutes?"
})

print("\n" + "="*50)
print("FINAL REPORT:")
print("="*50)
print(result["output"])




> Entering new AgentExecutor chain...
To determine which vehicles should be prioritized for immediate action and why, and to provide recommendations for the operations manager, I need to analyze the current state of the fleet vehicles. This involves identifying any anomalies or issues that require attention. 

Action: detect_anomalies
Action Input: fleet_data (assuming this is a string containing the current states of all fleet vehicles)ZX004: CRITICAL - Battery at 8%, immediate charging required
ZX004: ALERT - Offline for 120 minutes, investigate immediately
ZX006: ALERT - In maintenance for 200 minutes, status update needed
ZX007: WARNING - Battery at 15% with passenger onboard
ZX008: INFO - Charging at 78%, charger can be released
ZX010: CRITICAL - Battery at 3%, immediate charging required
ZX010: ALERT - Offline for 300 minutes, investigate immediatelyNow that I have the anomalies detected, I can see that there are several vehicles that require immediate attention. Vehicles ZX004